# Sensitivity Study: Transverse transport approximation

**Authors:** Jorrit Bakker, Alraune Zech

This notebooks investigates the validity of different formulations of the equational terms for transport in transverse direction in the approximate solutions. Alongside, you'll also see how to adapt/setup own models in the *mibitrans* package. 

## Background

### Governing equation
The governing equation for advective-dispersive transport with adsorption and decay given by *Bear (1979)*$^1$ is:

\begin{equation}\tag{1}
    R\frac{\partial C}{\partial t} = -v\frac{\partial C}{\partial x} + D_{x}\frac{\partial ^2 C}{\partial x^2} + D_y \frac{\partial^2 C}{\partial y^2} + D_z \frac{\partial^2 C}{\partial z^2} - \mu C
\end{equation}

With $0\leq x \leq \infty$, $-\infty \leq y \leq \infty$ and $-\infty \leq z \leq \infty$. With the contaminant source modeled as a continuous planar block located at x=0 and of width Y and thickness Z.

### Exact (`Mibitrans`) solution

The following exact analytical solution to the ADE was presented by *Wexler (1992)*$^2$:

\begin{align}\tag{2}
    C(x,y,z,t) = & \frac{C_{0}x}{8\sqrt{\pi \alpha_x v/R}}
    \int_0^t \tau^{-3/2} \exp \left(-\mu\tau - \frac{(x-v\tau/R)^2}{4\alpha_xv\tau/R}\right) \cdot\\
    & \quad \left[\operatorname{erf} \left( \frac{y+Y/2}{2\sqrt{D_y\tau/R}} \right) -
    \operatorname{erf} \left( \frac{y-Y/2}{2\sqrt{D_y\tau/R}} \right)\right]\cdot \nonumber \\
    & \quad\left[\operatorname{erf} \left( \frac{z+Z/2}{2\sqrt{D_z\tau/R}} \right) -
    \operatorname{erf} \left( \frac{z-Z/2}{2\sqrt{D_z\tau/R}} \right)\right] d\tau
    \nonumber
\end{align}

Where $\tau$ is the integration variable for time. There is no closed-form analytical solution, but the solution requires numerical integration. This equation (including numerical integration) is implemented in the `Mibitrans` model class.

### Anatrans solution

The exact solution can be simplified into a closed-form analytical equation by splitting the effect of time on the different directions following the principle $C(x,y,z,t)/C_0 = C(x,t)\cdot C(y,x)\cdot C(z,x)/C_0$. Specifically, by applying the substitution $\tau = x/v$ (time traveled of advection front) in the error function (`erc`) terms for the $y$ and $z$ directions. The remaining integral in $\tau$ in for the transport in $x$-direction can be solved analytically. The resulting approximate transport solution is:

\begin{align}\tag{3}
    C(x, y, z, t) &= \sum_{i=1}^{n}\left\{ \frac{C^*_{0,i}}{8} \exp \left(-\gamma_s t\right) \right. \\
    &\quad \cdot \left( \exp \left[ \frac{x\left(1-P\right)}{2\alpha_x}\right]
    \cdot \operatorname{erfc} \left[ \frac{x - Pv_Rt}{2\sqrt{\alpha_x v_Rt }} \right] \right.\\
    &\quad \:  + \left.  \exp \left[ \frac{x\left(1+P\right)}{2\alpha_x}\right]
    \cdot \operatorname{erfc} \left[ \frac{x + Pv_Rt}{2\sqrt{\alpha_x v_Rt }} \right] \right) \\
    &\quad \cdot \left( \operatorname{erf} \left[ \frac{y + Y_i}{2\sqrt{\alpha_y x}} \right] - \operatorname{erf} \left[ \frac{y - Y_i}{2\sqrt{\alpha_y x)}} \right] \right) \\
    &\quad \cdot \left. \left( \operatorname{erf} \left[ \frac{Z}{2\sqrt{\alpha_z x)}} \right] - \operatorname{erf} \left[ \frac{-Z}{2\sqrt{\alpha_z x}} \right] \right) \right\} \\
    & \text{with} \quad P = \sqrt{1+4\left(\mu - \gamma_s \right) \alpha_x/v_R}
\end{align}

### Alternative Approximation by *West et al, 2007*

*West et al., (2007)*$^3$ argue that the approximate solution to Eq. (2) should read differently with $\tau$ replaced by $t$ instead of performing the substitution of $\tau = x/v$ in the error function (`erc`) terms for the $y$ and $z$ directions, based on approximation steps outlined in *Domenico, (1987)*$^4$.
The equation, called the "final approximate nontruncated solution" (Equation 22 in *West et al., (2007)*$^3$) reads in its adaption to the notation and boundary conditions used in *mibitrans*:
\begin{align}\tag{4}
    C(x, y, z, t) &= \frac{C_{0}}{8} \left( \exp \left[ \frac{xv_R\left(1-P\right)}{2D_x/R}\right] 
    \cdot \operatorname{erfc} \left[ \frac{x - Pv_Rt}{2\sqrt{D_x t/R }} \right] \right. \\
     &\quad \:  + \left.  \exp \left[ \frac{xv_R\left(1+P\right)}{2D_x}\right] \\
     \cdot \operatorname{erfc} \left[ \frac{x + Pv_Rt}{2\sqrt{D_x t/R }} \right] \right) \\
    &\quad \cdot \left( \operatorname{erf} \left[ \frac{y + Y}{2\sqrt{D_y t / R}} \right] - \operatorname{erf} \left[ \frac{y - Y}{2\sqrt{D_y t / R)}} \right] \right) \\
    &\quad \cdot \left( \operatorname{erf} \left[ \frac{z + Z}{2\sqrt{D_z t / R)}} \right] - \operatorname{erf} \left[ \frac{z - Z}{2\sqrt{D_z t / R}} \right] \right) \\
    & \text{with} \quad P = \sqrt{1+4\mu R D_x/v^2}
\end{align}
This equation does not contain the effect of source depletion (i.e. $\gamma_s=0$) and a single source width (i.e. $n=i=1$). Equation (4) transforms into Eq (3) (without source depletion and single source) when sustituting $D_x = \alpha_L v$, $D_y = \alpha_y v = \alpha_y x/t$ and $D_z = \alpha_z v = \alpha_z x/t$.

Note that this equation gives clear indication to not be a correct approximation for 3D transport under the specified intial and boundary conditions as is does not conserve mass. For a continuous source (absent any source depletion) the following relation has to hold:
$$ \lim_{t \to \infty}C(x=0,y,z,t) = C_0 $$
For equation (4), $\lim_{t \to \infty}C(x=0,y,z,t) = 0$, as both transverse dispersion terms converge to zero for $t \to \infty$. Consequently the "final approximate nontruncated solution" of *West et al., (2007)*$^3$ is not a correct approximate solution in the first place. 

We use the versatile implementation of the *mibitrans* package to study this "final approximate non-truncated solution" (4) and confirm our observations. We'll implement it into a separate model class (also show-casing how to implement own models) and compare transport results with the implemented models `Mibitrans` (Eq. 2) and `Anatrans` (Eq. 3).

### References

$^1$ Bear,J., (1979) Hydraulics of groundwater, London ; New York : McGraw-Hill International Book Co.

$^2$ Wexler, E. J., (1992),  Analytical solutions for one-, two-, and three-dimensional solute transport in ground-water systems with uniform flow, Tech. Rep. 03-B7, U.S. G.P.O.

$^3$ [West, M. R., B. H. Kueper, and M. J. Ungs, (2007), On the use and error of approximation in the Domenico (1987) solution, Groundwater, 45 (2), 126–135.](https://doi.org/10.1111/j.1745-6584.2006.00280.x)

$^4$ [Domenico, P. A., 1987, An analytical model for multidimensional transport of a decaying contaminant
species, Journal of Hydrology, 91 (1), 49–58.](https://doi.org/10.1016/0022-1694(87)90127-2)


## Simulation and Model Comparison

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.special import erf
import mibitrans as mbt

### Costumize Model

We create a new model class for the alternative approximation by *West et al, 2007*$^1$ ("final approximate nontruncated solution"). Starting point is the `Anatrans` model class (Eq. 3), but we will redefined the handling of the transverse transport terms (as in Eq. 4). 

We start with defining a custom class (named `AnatransAlternativeSource`) and initialize it with the same arguments as the `Anatrans` model class. It inherrits all the functionalities of `Anatrans` by assigning it as the parent class by putting it between parentheses behind the class name. 

In a second step, we adapt those parts that we want to be different from the of `Anatrans` model class. We do so by redefining the handling of the source depletion function `_equation_term_source_depletion()` and `_equation_decay_sqrt()`. 

In [ ]:
class AnatransWithoutSubstitution(mbt.Anatrans):
    """Anatrans model class without substitution.
        Corresponds to approximate non-truncated solution (22) of West et al., 2007
    """

    def _equation_term_z(self, ttt):
        inner_term = self._src_pars.depth / (2 * np.sqrt(self._hyd_pars.alpha_z * self.rv * ttt))
        return erf(inner_term) - erf(-inner_term)

    def _equation_term_y(self, i, yyy, ttt):
        div_term = 2 * np.sqrt(self._hyd_pars.alpha_y * self.rv * ttt)
        term = erf((yyy + self.y_source[i]) / div_term) - erf((yyy - self.y_source[i]) / div_term)
        term[np.isnan(term)] = 0
        return term

    def _calculate_concentration_for_all_xyt(self, xxx, yyy, ttt):
        cxyt = 0
        decay_sqrt = np.sqrt(1 + 4 * (self._decay_rate - self.k_source) * self._hyd_pars.alpha_x / self.rv)
        x_term = self._equation_term_x(xxx, ttt, decay_sqrt)
        additional_x = self._equation_term_additional_x(xxx, ttt, decay_sqrt)
        z_term = self._equation_term_z(ttt)
        source_decay = self._equation_term_source_depletion(xxx, ttt)
        for i in range(len(self.c_source)):
            y_term = self._equation_term_y(i, yyy,ttt)
            cxyt_step = 1 / 8 * self.c_source[i] * source_decay * (x_term + additional_x) * y_term * z_term
            cxyt += cxyt_step
        if self._mode == "instant_reaction":
            self.cxyt_noBC = cxyt.copy()
            cxyt -= self.biodegradation_capacity
            cxyt = np.where(cxyt < 0, 0, cxyt)
        self.has_run = True
        return cxyt

### Setup Data Input

Parameters correspond to values of standard examples.

In [ ]:
hydro = mbt.HydrologicalParameters(velocity = 0.1, #0.1, 
                                   porosity = 0.3, #0.3,
                                   alpha_x = 3, 
                                   alpha_y = 0.02,
                                   alpha_z = 0.005)
att = mbt.AttenuationParameters(retardation = 1.0, 
                                half_life = 0)
source = mbt.SourceParameters(source_zone_boundary = [15], 
                              source_zone_concentration = [10],
                              depth = 2,
                            )
model = mbt.ModelParameters(model_length = 500,
                            model_width = 60,
                            model_time = 10 * 365)

### Model run

In [ ]:
mbt_object = mbt.Mibitrans(hydro, att, source, model)
mbt_results = mbt_object.run()

ana_object = mbt.Anatrans(hydro, att, source, model)
ana_results = ana_object.run()

alt_disp_object = AnatransWithoutSubstitution(hydro, att, source, model)
alt_disp_results = alt_disp_object.run()

### Visualization of results

Let's start with a visualization of the entire plume for the last step of the simutions for the `Anatrans` and the `Alternative` model class.

In [ ]:
ana3D = ana_results.plume_3d(cmap="cividis")
ana3D.set_zlim(0, 10)

alt3D = alt_disp_results.plume_3d(cmap="cividis")
alt3D.set_zlim(0, 10)
alt3D.set_title("Alternative Model")

Let's visualize the concentration differences between models along the plume center and at a cross section throught the plume:

In [ ]:
### Plot specifications
cmap = plt.get_cmap('tab20b')
colors_1 = cmap.colors   # list of RGB tuples

cmap = plt.get_cmap('Set1')
colors_2 = cmap.colors   # list of RGB tuples

ls_order = ['-','--',':','-.']
textsize = 8
lw = 2

In [ ]:
### Time and location for plots
t1 = 3*365 
t2 = 10*365 #3y\
x_pos = 75

In [ ]:
plt.figure(figsize = [3.75,2.5])
mbt_results.centerline(time = t1, color=colors_1[3],ls = '-',lw=lw, label="Mibitrans: $t_1$")
ana_results.centerline(time = t1, color=colors_1[3+8],ls = '--',lw=lw, label="Anatrans: $t_1$")
alt_disp_results.centerline(time = t1, color=colors_1[3+16],ls = ':',lw=lw+1, label="Alternative: $t_1$")

mbt_results.centerline(time=t2, color=colors_1[1],ls = '-',lw=lw, label=f"$t_2$")
ana_results.centerline(time=t2, color=colors_1[1+8],ls = '--',lw=lw, label=f"$t_2$")
alt_disp_results.centerline(time=t2, color=colors_1[1+16],ls = ':',lw=lw+1, label=f"$t_2$")

plt.legend(ncols = 2, fontsize = textsize, loc= 'lower left')#bbox_to_anchor=(0.3,0.55))
plt.xlabel('Plume travel distance $x$',fontsize = textsize)
plt.ylabel('Concentration $C(x,y=0,t_i)$',fontsize = textsize)
plt.title("")#Plume mass over time")
plt.xlim([0,500])
plt.ylim([0,10.2])
plt.grid(True)
plt.tick_params(axis='both', labelsize=textsize)  
plt.tight_layout()
#plt.savefig('model_substitution.pdf')
plt.show()

In [ ]:
plt.figure(figsize = [3.75,2.5])

mbt_results.transverse(time=t1, x_position= x_pos, color=colors_1[3],ls = '-',lw=lw, label="Mibitrans: $t_1$")
ana_results.transverse(time=t1, x_position= x_pos, color=colors_1[3+8],ls = '--',lw=lw, label="Anatrans: $t_1$")
alt_disp_results.transverse(time = t1,x_position= x_pos, color=colors_1[3+16],ls = ':',lw=lw+1, label="Alternative: $t_1$")

mbt_results.transverse(time=t2,x_position= x_pos, color=colors_1[1],ls = '-',lw=lw, label=f"$t_2$")
ana_results.transverse(time=t2,x_position= x_pos, color=colors_1[1+8],ls = '--',lw=lw, label=f"$t_2$")
alt_disp_results.transverse(time=t2,x_position= x_pos, color=colors_1[1+16],ls = ':',lw=lw+1, label=f"$t_2$")

plt.grid(True)
plt.ylim(0, 10.2)
plt.xlim([-25,22])
plt.title("Transverse plot of various models, at x={}m".format(x_pos))
plt.legend(ncols = 2, fontsize = textsize, loc= 'lower center')
plt.show()

## Conclusion

We see that the alternative model is not correct in its transport behavior.  As these graphs show, equation (4) deviates significantly from the exact solution (Eq 2), especially for larger times. This corresponds to the observation that Equation 5 is not conserving mass (i.e. no fullfilling boundary condition $\lim_{t \to \infty}C(x=0,y,z,t) = C_0$).

Note that also the `Bioscreen` model does also violate this condition at early times due to term truncation. We do not show these results here, but they can be found in the notebook `comparison_models_mbt_ana_bio.ipynb`.